# Lead Generation Analysis - Expected Value Simulation

Analyzing the expected value of purchasing leads from Sales Leads ABC Quizzes.

**Goal:** Determine average expected value and show the distribution (high/low scenarios)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load and Explore the Data

In [ ]:
# Load the lead generation data
df = pd.read_excel('Sales Leads ABC from Quizzes.xlsx', skiprows=1)

# Select relevant columns
df_clean = df[['id', 'Source', 'Purchase', 'Profit']].copy()

print("Dataset Overview:")
print(f"Total Leads: {len(df_clean):,}")
print(f"\nData Shape: {df_clean.shape}")
print("\nFirst few rows:")
df_clean.head(10)

## Expected Value Analysis

In [ ]:
# Calculate key statistics
avg_profit = df_clean['Profit'].mean()
median_profit = df_clean['Profit'].median()
min_profit = df_clean['Profit'].min()
max_profit = df_clean['Profit'].max()
std_profit = df_clean['Profit'].std()

# Calculate percentiles
p25 = df_clean['Profit'].quantile(0.25)
p75 = df_clean['Profit'].quantile(0.75)
p10 = df_clean['Profit'].quantile(0.10)
p90 = df_clean['Profit'].quantile(0.90)

print("="*60)
print("EXPECTED VALUE ANALYSIS")
print("="*60)
print(f"\n📊 Average Expected Value: ${avg_profit:.2f}")
print(f"📊 Median Expected Value:  ${median_profit:.2f}")
print(f"\n📈 Maximum Profit:         ${max_profit:.2f}")
print(f"📉 Minimum Profit:         ${min_profit:.2f}")
print(f"\n📊 Standard Deviation:     ${std_profit:.2f}")
print(f"\n🎯 10th Percentile (Low):  ${p10:.2f}")
print(f"🎯 25th Percentile:        ${p25:.2f}")
print(f"🎯 75th Percentile:        ${p75:.2f}")
print(f"🎯 90th Percentile (High): ${p90:.2f}")
print("\n" + "="*60)

## Conversion Analysis

In [ ]:
# Analyze purchase behavior
purchase_counts = df_clean['Purchase'].value_counts()
conversion_rate = (purchase_counts.get('Buy', 0) / len(df_clean)) * 100

# Calculate average profit for buyers vs non-buyers
buyers = df_clean[df_clean['Purchase'] == 'Buy']
non_buyers = df_clean[df_clean['Purchase'] == 'No Buy']

print("\n📊 CONVERSION METRICS")
print("="*60)
print(f"Total Leads: {len(df_clean):,}")
print(f"Conversions (Buy): {len(buyers):,} ({conversion_rate:.1f}%)")
print(f"No Purchase: {len(non_buyers):,} ({100-conversion_rate:.1f}%)")
print(f"\nAverage Profit (Buyers): ${buyers['Profit'].mean():.2f}")
print(f"Average Cost (Non-Buyers): ${non_buyers['Profit'].mean():.2f}")
print("="*60)

## Histogram: Distribution of Expected Values

In [ ]:
# Create comprehensive histogram
fig, ax = plt.subplots(figsize=(14, 8))

# Plot histogram
n, bins, patches = ax.hist(df_clean['Profit'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')

# Add vertical lines for key statistics
ax.axvline(avg_profit, color='red', linestyle='--', linewidth=2, label=f'Average: ${avg_profit:.2f}')
ax.axvline(median_profit, color='green', linestyle='--', linewidth=2, label=f'Median: ${median_profit:.2f}')
ax.axvline(p10, color='orange', linestyle=':', linewidth=2, label=f'10th %ile (Low): ${p10:.2f}')
ax.axvline(p90, color='purple', linestyle=':', linewidth=2, label=f'90th %ile (High): ${p90:.2f}')

# Formatting
ax.set_xlabel('Profit per Lead ($)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Lead Generation Expected Value Distribution\nSimulation of Buying Sales Leads', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

# Add text box with summary stats
textstr = f'''Expected Value Summary:
Average: ${avg_profit:.2f}
Median: ${median_profit:.2f}
Range: ${min_profit:.2f} to ${max_profit:.2f}
Std Dev: ${std_profit:.2f}
Conversion Rate: {conversion_rate:.1f}%'''

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("\n✅ Histogram generated successfully!")

## Analysis by Lead Source

In [ ]:
# Compare performance across different lead sources
source_analysis = df_clean.groupby('Source')['Profit'].agg([
    ('Count', 'count'),
    ('Average', 'mean'),
    ('Median', 'median'),
    ('Min', 'min'),
    ('Max', 'max'),
    ('Std Dev', 'std')
]).round(2)

print("\n📊 ANALYSIS BY LEAD SOURCE")
print("="*60)
print(source_analysis)
print("="*60)

In [ ]:
# Visualize by source
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by source
df_clean.boxplot(column='Profit', by='Source', ax=axes[0])
axes[0].set_title('Profit Distribution by Lead Source', fontweight='bold')
axes[0].set_xlabel('Lead Source', fontweight='bold')
axes[0].set_ylabel('Profit ($)', fontweight='bold')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
plt.sca(axes[0])
plt.xticks(rotation=0)

# Bar chart of average profit by source
source_avg = df_clean.groupby('Source')['Profit'].mean()
colors = ['green' if x > 0 else 'red' for x in source_avg.values]
source_avg.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Average Expected Value by Lead Source', fontweight='bold')
axes[1].set_xlabel('Lead Source', fontweight='bold')
axes[1].set_ylabel('Average Profit ($)', fontweight='bold')
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=0)

# Add value labels on bars
for i, v in enumerate(source_avg.values):
    axes[1].text(i, v + (5 if v > 0 else -15), f'${v:.2f}', 
                ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Key Insights & Recommendations

In [ ]:
print("\n" + "="*60)
print("KEY INSIGHTS")
print("="*60)

print(f"\n1. EXPECTED VALUE:")
print(f"   - Average expected profit per lead: ${avg_profit:.2f}")
print(f"   - This means on average, buying leads is {'PROFITABLE' if avg_profit > 0 else 'NOT PROFITABLE'}")

print(f"\n2. RISK ASSESSMENT:")
print(f"   - 10% chance of losing more than ${abs(p10):.2f} per lead")
print(f"   - 10% chance of making more than ${p90:.2f} per lead")
print(f"   - Worst case scenario: ${min_profit:.2f} loss")
print(f"   - Best case scenario: ${max_profit:.2f} profit")

print(f"\n3. CONVERSION METRICS:")
print(f"   - Only {conversion_rate:.1f}% of leads convert to purchases")
print(f"   - When they DO convert, average profit is ${buyers['Profit'].mean():.2f}")
print(f"   - When they DON'T convert, average loss is ${abs(non_buyers['Profit'].mean()):.2f}")

print(f"\n4. VOLUME CONSIDERATIONS:")
if avg_profit > 0:
    leads_for_1000 = 1000 / avg_profit
    print(f"   - To make $1,000 profit, you'd need ~{leads_for_1000:.0f} leads on average")
    print(f"   - At ${avg_profit:.2f} per lead, buying 100 leads = ${avg_profit * 100:.2f} expected profit")
else:
    print(f"   - With negative expected value, buying more leads = more losses")

print("\n" + "="*60)